# Pipeline: Install and Set Up VS Code + Python + Essential Extensions

This notebook gives you a reproducible setup pipeline for Windows, macOS, and Linux using command-based steps.

## How to Use
1. Run cells in order.
2. If a command fails, read the troubleshooting notes in that section.
3. Prefer running terminal commands from VS Code integrated terminal when prompted.

## 1) Detect OS and Prerequisites

This section detects your platform and checks if common package managers are available:
- Windows: `winget`
- macOS: `brew`
- Linux: `apt`

It also checks terminal basics and whether the `code` CLI is already available.

In [ ]:
import platform
import shutil
import subprocess

os_name = platform.system()
shell = "PowerShell" if os_name == "Windows" else "Bash/Zsh"

print(f"Detected OS: {os_name}")
print(f"Recommended shell: {shell}")

for tool in ["winget", "brew", "apt", "code", "python", "python3", "py", "pip", "pip3"]:
    path = shutil.which(tool)
    print(f"{tool:8} -> {'FOUND: ' + path if path else 'not found'}")

print("\nTip: If `code` is not found, open VS Code Command Palette and run: 'Shell Command: Install code command in PATH' (macOS/Linux) or reinstall VS Code with PATH integration (Windows).")

## 2) Install Visual Studio Code from CLI

Run the command for your OS in a terminal.

### Windows (PowerShell)
```powershell
winget install --id Microsoft.VisualStudioCode -e --source winget
```

### macOS (Terminal)
```bash
brew install --cask visual-studio-code
```

### Ubuntu/Debian (Terminal)
```bash
sudo apt update
sudo apt install -y code
```

After install, verify:
```bash
code --version
```

In [ ]:
import shutil
import subprocess

if shutil.which("code"):
    try:
        result = subprocess.run(["code", "--version"], capture_output=True, text=True, check=False)
        print(result.stdout if result.stdout else result.stderr)
    except Exception as exc:
        print(f"Could not run code --version: {exc}")
else:
    print("`code` command not found yet. Install VS Code and ensure PATH integration is enabled.")

## 3) Install Python and Validate `pip`

Use one option for your OS.

### Windows (PowerShell)
```powershell
winget install --id Python.Python.3.12 -e
```

### macOS (Terminal)
```bash
brew install python
```

### Ubuntu/Debian (Terminal)
```bash
sudo apt update
sudo apt install -y python3 python3-pip python3-venv
```

Validation commands:
```bash
python --version
python3 --version
py --version
pip --version
pip3 --version
```

In [ ]:
import shutil
import subprocess

commands = [
    ["python", "--version"],
    ["python3", "--version"],
    ["py", "--version"],
    ["pip", "--version"],
    ["pip3", "--version"],
]

for cmd in commands:
    if shutil.which(cmd[0]):
        result = subprocess.run(cmd, capture_output=True, text=True, check=False)
        output = (result.stdout or result.stderr).strip()
        print(f"{' '.join(cmd):18} -> {output}")
    else:
        print(f"{' '.join(cmd):18} -> command not found")

## 4) Create Project Folder and Virtual Environment

This step creates a clean workspace and Python virtual environment.

### Commands (choose your shell)

#### PowerShell (Windows)
```powershell
mkdir dev_workspace
cd dev_workspace
python -m venv .venv
.\.venv\Scripts\Activate.ps1
```

#### Bash/Zsh (macOS/Linux)
```bash
mkdir -p dev_workspace
cd dev_workspace
python3 -m venv .venv
source .venv/bin/activate
```

Expected result: your prompt indicates `(.venv)` is active.

In [ ]:
import os
import platform
from pathlib import Path

project_dir = Path.cwd() / "dev_workspace"
project_dir.mkdir(exist_ok=True)
os.chdir(project_dir)

print(f"Current project folder: {project_dir}")

if platform.system() == "Windows":
    print("Run in terminal: python -m venv .venv")
    print("Then activate: .\\.venv\\Scripts\\Activate.ps1")
else:
    print("Run in terminal: python3 -m venv .venv")
    print("Then activate: source .venv/bin/activate")

## 5) Install Core Python Developer Packages

With the virtual environment active, install common developer tooling and freeze dependencies.

```bash
python -m pip install --upgrade pip
pip install ipykernel pytest black flake8 pylint mypy
pip freeze > requirements.txt
```

In [ ]:
from pathlib import Path

requirements_file = Path("requirements.txt")
print("Run these commands in your active terminal:\n")
print("python -m pip install --upgrade pip")
print("pip install ipykernel pytest black flake8 pylint mypy")
print("pip freeze > requirements.txt")

if requirements_file.exists():
    print("\nrequirements.txt already exists. Top lines:")
    print("\n".join(requirements_file.read_text(encoding="utf-8").splitlines()[:10]))
else:
    print("\nrequirements.txt not found yet. Create it after package install.")

## 6) Install VS Code Extensions with `code --install-extension`

Install core Python extensions:
- `ms-python.python`
- `ms-python.vscode-pylance`
- `ms-toolsai.jupyter`
- `ms-python.debugpy`
- `ms-python.black-formatter`
- `ms-python.isort`
- `ms-python.flake8`

Commands:
```bash
code --install-extension ms-python.python
code --install-extension ms-python.vscode-pylance
code --install-extension ms-toolsai.jupyter
code --install-extension ms-python.debugpy
code --install-extension ms-python.black-formatter
code --install-extension ms-python.isort
code --install-extension ms-python.flake8
code --list-extensions
```

In [ ]:
import shutil
import subprocess

extensions = [
    "ms-python.python",
    "ms-python.vscode-pylance",
    "ms-toolsai.jupyter",
    "ms-python.debugpy",
    "ms-python.black-formatter",
    "ms-python.isort",
    "ms-python.flake8",
]

if not shutil.which("code"):
    print("`code` CLI not found. Install VS Code and add CLI to PATH before running this cell.")
else:
    for ext in extensions:
        cmd = ["code", "--install-extension", ext]
        print(f"Installing {ext}...")
        subprocess.run(cmd, check=False)

    print("\nInstalled extensions:")
    subprocess.run(["code", "--list-extensions"], check=False)

## 7) Create Workspace Settings for Python in VS Code

This writes `.vscode/settings.json` with interpreter, formatting, linting, pytest discovery, and notebook consistency settings.

In [ ]:
import json
import platform
from pathlib import Path

vscode_dir = Path(".vscode")
vscode_dir.mkdir(exist_ok=True)
settings_path = vscode_dir / "settings.json"

interpreter = "${workspaceFolder}/.venv/Scripts/python.exe"
if platform.system() != "Windows":
    interpreter = "${workspaceFolder}/.venv/bin/python"

settings = {
    "python.defaultInterpreterPath": interpreter,
    "python.terminal.activateEnvironment": True,
    "python.testing.pytestEnabled": True,
    "python.testing.unittestEnabled": False,
    "python.testing.pytestArgs": ["tests"],
    "python.linting.enabled": True,
    "editor.formatOnSave": True,
    "python.analysis.typeCheckingMode": "basic",
    "[python]": {
        "editor.defaultFormatter": "ms-python.black-formatter"
    },
    "jupyter.askForKernelRestart": False,
    "notebook.formatOnSave.enabled": True
}

settings_path.write_text(json.dumps(settings, indent=2), encoding="utf-8")
print(f"Created: {settings_path}")
print(settings_path.read_text(encoding='utf-8'))

## 8) Run Verification Script and Unit Tests in VS Code

This section creates:
- `hello_setup.py` with package/version checks
- `tests/test_setup.py` with a basic pytest test

Then it runs:
```bash
python hello_setup.py
pytest -q
```

In [ ]:
from pathlib import Path
import subprocess
import sys

hello_script = Path("hello_setup.py")
tests_dir = Path("tests")
tests_dir.mkdir(exist_ok=True)
test_file = tests_dir / "test_setup.py"

hello_script.write_text(
    """import sys\n\nmodules = [\"pytest\", \"black\", \"flake8\", \"pylint\", \"mypy\"]\nprint(f\"Python: {sys.version}\")\n\nfor m in modules:\n    try:\n        __import__(m)\n        print(f\"[OK] {m}\")\n    except Exception as exc:\n        print(f\"[MISSING/ERROR] {m}: {exc}\")\n""",
    encoding="utf-8",
)

test_file.write_text(
    """def test_basic_math():\n    assert 2 + 2 == 4\n""",
    encoding="utf-8",
)

print(f"Created {hello_script}")
print(f"Created {test_file}")

print("\nRunning verification script...\n")
subprocess.run([sys.executable, str(hello_script)], check=False)

print("\nRunning pytest...\n")
subprocess.run([sys.executable, "-m", "pytest", "-q"], check=False)

## Troubleshooting

1. `python` not found:
   - Reopen terminal after install.
   - On Windows, reinstall Python and check "Add Python to PATH".
2. PowerShell cannot activate venv:
   - Run `Set-ExecutionPolicy -Scope CurrentUser RemoteSigned` once, then reopen terminal.
3. VS Code using wrong interpreter:
   - Press `Ctrl+Shift+P` -> `Python: Select Interpreter` -> choose `.venv`.
4. `code` command not found:
   - Reinstall VS Code with PATH option, or configure shell command from VS Code.

## Quick Daily Workflow

1. Open project folder in VS Code.
2. Open terminal and activate environment (`.venv`).
3. Run scripts/notebooks.
4. Format code with Black on save.
5. Run tests with `pytest -q` before commits.